## Requirements
Open terminal and type: pip install -r requirements.txt

For actually placing orders on Kalshi you will need to:  
    - Add your Kalshi API key as a file.  Default name Prod_Key.txt  
    - Add your Kalshi Access Key as access_key (in Parameters section)  

## Imports

In [1]:
import requests
import time
import json
import re
import uuid
import datetime
import base64
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.backends import default_backend

## ESPN Functions

In [2]:
def get_live_games():
    """Return list of live NFL game id's from ESPN"""
    url = "https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard"
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        data = r.json()
        games = []
        for event in data.get("events", []):
            state = event.get("status", {}).get("type", {}).get("state")
            if state == "in":
                games.append(event["id"])
        return games
    except Exception as e:
        print("ERROR: get_live_games:", e)
        return []

In [3]:
def get_plays_from_game(game_id):
    """Return all previous plays from an ESPN game id"""
    url = f"https://site.api.espn.com/apis/site/v2/sports/football/nfl/summary?event={game_id}"
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        data = r.json()
        drives = data.get("drives", {}).get("previous", [])
        plays = []
        for drive in drives:
            plays.extend(drive.get("plays", []))
        return plays
    except Exception as e:
        print(f"ERROR: get_plays_from_game({game_id}):", e)
        return []

In [4]:
def get_scorer_from_play(play):
    """Extract the last name of the player who scored the touchdown"""
    text = play.get("text", "")
    play_type = play.get("type", {}).get("text", "").lower()

    ## Passing / Receiving TD: look after "to"
    if "pass" in play_type or "receiving" in play_type:
        match = re.search(r"\bto\s+([A-Z]\.[A-Za-z'-]+)", text)
        if match:
            return match.group(1).split('.')[-1]

    ## Rushing TD: look before "TOUCHDOWN" 
    if "rush" in play_type or "rushing" in play_type:
        match = re.search(r"([A-Z]\.[A-Za-z'-]+)\s+.*TOUCHDOWN", text)
        if match:
            return match.group(1).split('.')[-1]

    ## Fallback: last capitalized word before "TOUCHDOWN", for special teams touchdowns
    words = text.replace(",", "").split()
    for i, w in enumerate(words):
        if "touchdown" in w.lower() and i > 0:
            for j in range(i-1, -1, -1):
                if words[j].istitle():
                    return words[j]

    return "Unknown"  ## Returning Unknown is fine because there aren't markets on Kalshi with Unknown

## Kalshi Functions

In [5]:
def get_kalshi_game_information(kalshi_game_id):
    """Get all infomation about the Kalshi market for a given NFL game"""
    url = f"https://api.elections.kalshi.com/trade-api/v2/markets?event_ticker={kalshi_game_id}&limit=1000"
    market_data = requests.get(url).json()
    return market_data

In [6]:
def load_private_key_from_file(file_path):
    """Loads your private API key to be used to post orders to Kalshi"""
    with open(file_path, "rb") as key_file:
        private_key = serialization.load_pem_private_key(
            key_file.read(),
            password=None,  # or provide a password if your key is encrypted
            backend=default_backend()
        )
    return private_key

In [7]:
def create_signature(private_key, timestamp, method, path):
    """Create the request signature required for Kalshi"""
    path_without_query = path.split('?')[0]
    message = f"{timestamp}{method}{path_without_query}".encode('utf-8')
    signature = private_key.sign(
        message,
        padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.DIGEST_LENGTH),
        hashes.SHA256()
    )
    return base64.b64encode(signature).decode('utf-8')

In [8]:
def post(private_key, api_key_id, path, data, base_url):
    """Make an authenticated POST request to the Kalshi API"""
    timestamp = str(int(datetime.datetime.now().timestamp() * 1000))
    signature = create_signature(private_key, timestamp, "POST", path)

    headers = {
        'KALSHI-ACCESS-KEY': api_key_id,
        'KALSHI-ACCESS-SIGNATURE': signature,
        'KALSHI-ACCESS-TIMESTAMP': timestamp,
        'Content-Type': 'application/json'
    }

    return requests.post(base_url + path, headers=headers, json=data)

## Parameters

In [9]:
## These are parameters that can be changed
private_key_location = 'Prod_Key.txt'  ## Your private key for prod environment
access_key = ''   ## Your Kalshi access key


CHECK_INTERVAL = 3  ## seconds between checks
purchase_quantity = 1000  ##quantity you would like to purchase
purchase_price = 96  ## max price you would like to purcahse at

In [ ]:
## There are parameters that should not be changed
seen_touchdowns = set()
BASE_URL = 'https://api.elections.kalshi.com'  ## used to change between prod and demo environments
add_url = '/trade-api/v2/portfolio/orders'  ## used to get to orders
private_key = load_private_key_from_file(private_key_location)

In [11]:
## this is for specific example.  The Game you want to follow
Example_Kalshi_Game = 'KXNFLANYTD-25OCT27WASKC'
market_data = get_kalshi_game_information(Example_Kalshi_Game)

## Main Definition

In [ ]:
def main():
    print("Starting NFL touchdown bet placer")
    global seen_touchdowns

    while True:
        try:
            live_games = get_live_games()  ## pull the ESPN live NFL games
            for game_id in live_games:
                plays = get_plays_from_game(game_id)  ## pull all plays for all live NFL games from ESPN
                for play in plays:
                    if play.get("scoringPlay"):
                        td_id = play.get("id")  ## pull only the plays where a touchdown has been scored from ESPN
                        if td_id not in seen_touchdowns:
                            seen_touchdowns.add(td_id)  ## make sure that it is a new td and not one that has already been flagged
                            scorer = get_scorer_from_play(play)  ## get the last name of the person who scored the touchdown

                            ticker = next((m["ticker"]for m in market_data['markets']if scorer.lower() in m.get("no_sub_title", "").lower()),None)  ## get specific market for scoring player

                            order_data = {"ticker": ticker,"action": "buy","side": "yes","count": purchase_quantity,"type": "limit","yes_price": purchase_price,"client_order_id": str(uuid.uuid4())}  ##order information to send
                            
                            response = post(private_key, access_key, add_url, order_data, base_url=BASE_URL)  ## place the order
                            if response.status_code == 201:
                                 order = response.json()['order']
                                 print(f"Order placed successfully!")
                                 print(f"Order ID: {order['order_id']}")
                                 print(f"Client Order ID: {order_data['client_order_id']}")
                                 print(f"Status: {order['status']}")
                            else:
                                print(f"Error: {response.status_code} - {response.text}")

                            print(ticker, scorer)

        except Exception as e: print("ERROR: main loop:", e)
        time.sleep(CHECK_INTERVAL)

## Running Main
Won't work if there are not active NFL games going on, as there is no game id to look at

In [ ]:
if __name__ == "__main__": main()

## Kalshi Side Example
For this example we will be using Washignton Commanders vs Kansas City Chiefs on October 27, 2025  
We are assuming we just got a notification that Rashee Rice scored a touchdown  

In [12]:
Example_Kalshi_Game = 'KXNFLANYTD-25OCT27WASKC'
Example_Kalshi_Game_Scorer = 'Rice'

In [ ]:
## Get Kalshi information from specified game
Example_Kalshi_Game_Information = get_kalshi_game_information(Example_Kalshi_Game)
Example_Kalshi_Game_Information

In [14]:
## Get the specific touchdown market for the scorer
Example_Scorer_Market = next((m["ticker"]for m in Example_Kalshi_Game_Information['markets']if Example_Kalshi_Game_Scorer.lower() in m.get("no_sub_title", "").lower()),None)
Example_Scorer_Market

'KXNFLANYTD-25OCT27WASKC-KCRRICE4'

In [15]:
## Get the order data that you want to submit  (I removed client_order_id for this example, but it is needed on Kalshi side)
order_data = {"ticker": Example_Scorer_Market,"action": "buy","side": "yes","count": purchase_quantity,"type": "limit","yes_price": purchase_price}
order_data

{'ticker': 'KXNFLANYTD-25OCT27WASKC-KCRRICE4',
 'action': 'buy',
 'side': 'yes',
 'count': 1000,
 'type': 'limit',
 'yes_price': 96}

In [ ]:
## this will post your order to Kalshi
post(private_key, access_key, add_url, order_data, base_url=BASE_URL)